In [ ]:
# Cài nnU-Net bản chuẩn từ PyPI (Nhanh và sạch)
!pip install -q nnunetv2

# Cài thêm các thư viện bổ trợ nếu thiếu
!pip install -q nibabel matplotlib medpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!nvidia-smi

Sun Mar  1 06:08:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   56C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# A. Set up

## 1. Giải nén `nnUNet_raw` & `nnUNet_preprocessed`

In [ ]:
import os
import zipfile
from tqdm import tqdm

RAW_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip"
PRE_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip"

RAW_DIR = "/content"
PRE_DIR = "/content"

def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for f in tqdm(z.namelist()):
            z.extract(f, out_dir)

In [ ]:
unzip(RAW_ZIP, RAW_DIR)

100%|██████████| 1848/1848 [02:10<00:00, 14.15it/s]


In [ ]:
unzip(PRE_ZIP, PRE_DIR)

100%|██████████| 2590/2590 [03:10<00:00, 13.59it/s]


## 2. Set biến môi trường nnU-Net v2

In [ ]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/nnUNet_results


# B. Inference Baseline

In [ ]:
# CHẾ ĐỘ 1: CHẠY MODEL EDL (UNCERTAINTY DECOMPOSITION)
# ------------------------------------------------------------------
# Script này sẽ thực hiện quy trình full:
# 1. Tự động "tiêm" (inject) file EDLTrainer.py vào thư viện nnU-Net hệ thống.
# 2. Load trọng số Model EDL (checkpoint_best.pth) từ đường dẫn config 'edl'.
# 3. Chạy Inference và tính toán toán học để tách Uncertainty thành:
#    - Aleatoric (Nhiễu dữ liệu)
#    - Epistemic (Mô hình chưa biết)
# 4. Lưu kết quả Segmentation + 3 bản đồ Uncertainty (.nii.gz).

In [ ]:
# Đổi tên file cho đúng chuẩn (giải nén còn thừa .gz)
!for f in /content/nnUNet_raw/Dataset101_BraTS2020/imagesTr/*.nii.gz; do mv "$f" "${f%.gz}"; done

## 1. Fold 0

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode baseline_raw --fold 0 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: BASELINE_RAW | FOLD: 0 ---
🔧 Initializing Engine | Mode: BASELINE...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 0: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_011...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii -> KHÔNG
100% 8/8 [00:02<00:00,  3.98it/s]
1        | BRATS_011       | 0.9184   | 0.9195   | 0.7720   | 0.8700
    📸 Drawing Slice: 112
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Data

## 2. Fold 1

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode baseline_raw --fold 1 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: BASELINE_RAW | FOLD: 1 ---
🔧 Initializing Engine | Mode: BASELINE...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 1: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_004...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_004.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_004.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.66it/s]
1        | BRATS_004       | 0.9515   | 0.9168   | 0.8874   | 0.9186
    📸 Drawing Slice: 98
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Datas

## 3. Fold 2

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode baseline_raw --fold 2 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: BASELINE_RAW | FOLD: 2 ---
🔧 Initializing Engine | Mode: BASELINE...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 2: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_002...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_002.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_002.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.73it/s]
1        | BRATS_002       | 0.9065   | 0.9602   | 0.8609   | 0.9092
    📸 Drawing Slice: 34
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Datas

## 4. Fold 3

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode baseline_raw --fold 3 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: BASELINE_RAW | FOLD: 3 ---
🔧 Initializing Engine | Mode: BASELINE...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 3: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_003...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_003.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_003.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.66it/s]
1        | BRATS_003       | 0.8309   | 0.9154   | 0.8597   | 0.8687
    📸 Drawing Slice: 94
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Datas

## 5. Fold 4

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode baseline_raw --fold 4 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: BASELINE_RAW | FOLD: 4 ---
🔧 Initializing Engine | Mode: BASELINE...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 4: 73 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 73 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_001...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_001.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_001.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.64it/s]
1        | BRATS_001       | 0.9198   | 0.9462   | 0.8920   | 0.9193
    📸 Drawing Slice: 44
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Datas